In [1]:
#Import required packages
import csv
import os
import datetime

NOTE, ALL THE PATH VARIABLES ARE WHAT WILL NEED ADJUSTING TO SPECIFIC DATASET (CHANNELS CAN ALSO BE MODIFIED IN CASE NHS CHANNEL IS NOT CHANNEL 2)

In [ ]:
#Defines path variable for directory containining all images
source_folder_all_images = 'PATH TO DIRECTORY WITH ALL RAW IMAGES'
#Defines path variable for directory where the symbolic link for slices that are focused on either bottom or middle are created
OUTPUT_FOLDER =  'PATH TO DESIRED OUTPUT DIRECTORY'

#Paths to the four .txt files that contain the FOV + slice information for bottom and middle of cells (in confluent/non-confluent images)
BOTTOM_FOVs_file = 'PATH TO .TXT FILE GENERATED IN '01' THAT CONTAINS THE NAMES OF THE FOCUS SLICES'
CONFLUENT_BOTTOM_FOVs_file = 'PATH TO .TXT FILE GENERATED IN '01' THAT CONTAINS THE NAMES OF IMAGES REMOVED DUE TO BEING TOO CONFLUENT'

In [3]:
#Defines variable all_images as a lift of all the filenames in the input image folder
all_images = os.listdir(source_folder_all_images)

In [4]:
#Creates a function that opens the .txt files with FOV + slice information and only reads in the substring of each file name that contains the FOV + slice information and makes it into a list and returns that (if the list is empty, then it doesn't do that)
def GetFocusFOVs(PATH):
    with open(PATH, newline= '') as f: #This opens the file specified by PATH variable as f
        reader = csv.reader(f) #csv.reader is then used to read the file as a .csv
        data = list(reader) #The csv object is then converted into list (really, it is a list of lists)
        FOV = [] #Creates empty list for iteration where the FOVs from the list can be put into
        if(len(data) > 0): #If the length of the list is more than 0, then the following is executed
            data = data[0] #Takes only the first list of lists (which corresponds to the first line in the .txt): this is the only line in the .txt that contains FOV + slice information
            for i in data: #For loop that iterates over each object in list and puts the substring (which corresponds to only FOV + splice info) into the FOV list.
                FOV.append(i[0:12])
        return FOV #Returns the FOV list as the output of the function


In [5]:
#Uses the defined function for extracting FOV + splice informtion on the specified .txts
BOTTOM_FOVs = GetFocusFOVs(BOTTOM_FOVs_file) 
CONFLUENT_BOTTOM_FOVs = GetFocusFOVs(CONFLUENT_BOTTOM_FOVs_file)

With lists of the FOV+slices that are in focus, we now create lists of all the images (from all four channels) that correspond to these FOV +slices

In [6]:
#Empty lists for iterations (to create the list of images in focus)
BOT_ALL_CHANNELS = []
CONFLUENT_BOT_ALL_CHANNELS = []

#The for loop below then goes through all the images lists in input image directory and checks their position corresponds to the FOV + slices in focus
for i in all_images:
    image = i #i saved as image to make it clear, although this isn't necesarry
    FOV_i = image[0:12] #FOV_i is a substring of the image name, that corresponds to the FOV + stack position. This is to be comapred with the FOV + stack position defined as in focus.
    if(FOV_i in BOTTOM_FOVs): #If the FOV + stack position of the image (FOV_i) is in the list of FOV + stack positions in focus for the BOTTOM of the cells (in non-confluent images), then i is appended BOTTOM_FOV list
        BOT_ALL_CHANNELS.append(image) #Image filenames meeting this requirement are BOT_ALL_CHANNELS
    if(FOV_i in CONFLUENT_BOTTOM_FOVs): #If FOV_i is present in CONFLUENT_BOTTOM_FOVs, then the imaged is appended in CONFLUENT_BOT_ALL_CHANNELS list
        CONFLUENT_BOT_ALL_CHANNELS.append(image)


To verify that all images that are selected as belonging to either BOT (non-confluent/confluent) or MID (non-confluent/confluent) corresponds to the FOVs of the segmentation channel

In [7]:
#Code below checks that all FOVs in the BOT_ch1_3_4 appears in FOV_BOT. Same goes for MID_FOVs. These are defined as variables (which are boolean, either True or False)
BOT_CORRECTLY_MATCHED = all(i[0:12] in BOTTOM_FOVs for i in BOT_ALL_CHANNELS)
CONFLUENT_BOT_CORRECTLY_MATCHED = all(i[0:12] in CONFLUENT_BOTTOM_FOVs for i in CONFLUENT_BOT_ALL_CHANNELS)


#The if/else checks the matching of the different FOVs and the images that are created above. For each FOV, it is checked whether the statement is True. If all are True, then the script can rune. Otherwise, a RunTimeError is raised indicating what set the pairing went wrong in
if(BOT_CORRECTLY_MATCHED): #The first if/eles relates to BOT FOVs
    print('FOVs of BOTTOM focused slices are correctly with images')
else:
    raise RuntimeError('IMPERFECT MATCH BETWEEN BOTTOM_FOVs and BOT_ALL_CHANNELS') from None
if(CONFLUENT_BOT_CORRECTLY_MATCHED): #The first if/eles relates to BOT FOVs
    print('FOVs of CONFLUENT_BOTTOM focused slices are correctly with images')
else:
    raise RuntimeError('IMPERFECT MATCH BETWEEN CONFLUENT_BOTTOM_FOVs and CONFLUENT_BOT_ALL_CHANNELS') from None

FOVs of BOTTOM focused slices are correctly with images
FOVs of CONFLUENT_BOTTOM focused slices are correctly with images


With FOV + stack positions being correctly paired, the following function then checks whether all FOV + stack positions have the same number of images per channel (as one image per channel should be present per position)

In [8]:
#The function takes the FOV as the input and creates empty variables for iteration for each channel. It then goes through all images defined as in focus for the given FOV and 
#appends the image to a channel variable depending on what channel the image comes from (the channel is defined through substring of the image name that corresponds to where the channel information is given)
#And if statement is then used to check if all channel lists are of the same length (meaning they contain the same no. of images)
#If True, it prints that an equal number of images across channels are present, otherwise a RunTimeError is raised.
def SameNumberOfImagesPerChannel(FOV_ALL_CHANNELS):
    ch1 = [] #Empty lists for iteration
    ch2 = [] #Empty lists for iteration
    ch3 = [] #Empty lists for iteration
    ch4 = [] #Empty lists for iteration
    for i in FOV_ALL_CHANNELS: #For all images that are not channel 2 in BOT FOCUSED IMAGES
        if(i[13:16] == 'ch1'): #If index 13-16 matches 'ch1', make that image part of the channel1_bot list
            ch1.append(i)
        if(i[13:16] == 'ch2'): #If index 13-16 matches 'ch1', make that image part of the channel1_bot list
            ch2.append(i)
        if(i[13:16] == 'ch3'): #If index 13-16 matches 'ch3', make that image part of the channel3_bot list
            ch3.append(i)
        if(i[13:16] == 'ch4'): #If index 13-16 matches 'ch4', make that image part of the channel4_bot list
            ch4.append(i)
    if(len(ch1) == len(ch2) & len(ch1) == len(ch3) & len(ch1) == len(ch4)): #If statement that requires the length of channel lists of channel 2, 3, and 4 to be equal in length to channel 1
        print('For SPECIFIC FOV (SEE FUNCTION INPUT) there is equal number of images from each channel') #If True, this is printed and script can run on
    else: #Else, a runtime error is raised
        raise RuntimeError('Difference in number of images per channels for SPECIFIC FOV (SEE FUNCTION INPUT) focused FOVs') from None

In [9]:
#The function is then run on the different FOVs
SameNumberOfImagesPerChannel(BOT_ALL_CHANNELS)
SameNumberOfImagesPerChannel(CONFLUENT_BOT_ALL_CHANNELS)

For SPECIFIC FOV (SEE FUNCTION INPUT) there is equal number of images from each channel
For SPECIFIC FOV (SEE FUNCTION INPUT) there is equal number of images from each channel


Now that we have selected the correct images of ALL CHANNELS (1,2,3,4) corresponds to the FOVs in focus, a symbolic link between the images and a new folder is created (this is to save processing time/space compared to copying the files)

In [10]:
#The for loop below does this (again split according to BOT and MID, although could be made to run as one using a list object containing the lists of BOT_ch1_ch3_ch4 and MID_ch1_ch3_ch4)
#The symbolic link is created using the following function that takes the following inputs:
    #The FOV set w. image names for all channels
    #A string specifying whether the output should be put in BOTTOM or MIDDLE folder. 
    #True/False boolean value for the Confluent variable, to indicate whether the FOVs should be saved in normal channel folders or the ones for confluent channels
def SymbolicLinkImages (FOV_CHANNELS, POSITION, CONFLUENT):
    begin_time = datetime.datetime.now() #Begin_time variable is used to define a starting timepoint for when the function is executed (used to measure computing time)
    for i in FOV_CHANNELS: #All images listed in the specified FOV are then treated as follows:
        original_file = os.path.join(source_folder_all_images, i) #This defines path to the original image file (source folder of all images defined at top of script). Note, os.path.join is used when fusing strings that are paths as the dashed lines can otherwise cause problems
        if(CONFLUENT == True): #If statement that takes the boolean value of the confluent variable, if set to True, the symbolic link is put in folder for confluent channel
            destination_folder = os.path.join(OUTPUT_FOLDER,(POSITION+'/CONFLUENT_CHANNEL'+i[15])) #Selects the destination folder (based on what the whether the BottomOrMiddle string is given and what channel number the image corresponds to). (OUTPUT FOLDER difined at top of script)
        else: #If confluent == False, the symbolic link is put in folder that is NOT for confluent channels    
            destination_folder = os.path.join(OUTPUT_FOLDER,(POSITION+'/CHANNEL'+i[15])) #Selects the destination folder (based on what the whether the BottomOrMiddle string is given and what channel number the image corresponds to). (OUTPUT FOLDER difined at top of script)
        destination_filename = i[0:12]+"_"+i[13:16]+".tiff" #Creates the filename that the symbolic link will have. It only changes the - to a _, as some command line tools freak out with -.
        destination_path = os.path.join(destination_folder, destination_filename) #The full destination_path (which is the input for the symbolic link funtion) is then made from destination folder and filename
        os.symlink(original_file, destination_path) #os.symlink is then called to create the symbolic link. NOTE, THIS REQUIRES ADMIN RIGHTS TO WORK
    print("Symbolic link for SPECIFIC FOV (SEE FUNCTION INPUT) finished after:") #After all symbolic links for a FOV is created, this is printed
    print(datetime.datetime.now() - begin_time) #And below the text, the time different between begin_time and time now is printed which is the measurement of how long it took to execute

In [11]:
#The SymbolicLinkImages function is run on all the FOVs (BOT_ALL_CHANNELS, CONFLUENT_BOT_ALL_CHANNELS, MID_ALL_CHANNELS, CONFLUENT_MID_ALL_CHANNELS)
SymbolicLinkImages(BOT_ALL_CHANNELS, 'BOTTOM', CONFLUENT = False)
SymbolicLinkImages(CONFLUENT_BOT_ALL_CHANNELS, 'BOTTOM', CONFLUENT = True)

Symbolic link for SPECIFIC FOV (SEE FUNCTION INPUT) finished after:
0:00:00.312752
Symbolic link for SPECIFIC FOV (SEE FUNCTION INPUT) finished after:
0:00:00.000005
